# Digitization of ECG Images

**Group Members:**
- Siddhnat sahu - 202301070159
- Aryan Kumar- 202301070164
- Amir Furquani - 202301070165
- Raviraj Sonar - 202301070167


In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from sklearn.model_selection import KFold, train_test_split
from scipy import signal as scipy_signal
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import gc

# Configuration
CONFIG = {
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'image_size': (5600, 1700),
    'crop_size': (5600, 1696),
    't0': 301,
    't1': 5301,
    'mv_to_pixel': 79.0,
    'zero_mv': [703.5, 987.5, 1271.5, 1531.5],
    'window_size': 240,
    'batch_size': 1,           # Reduced to 1 for 6GB GPU
    'accumulation_steps': 4,   # Simulate batch size 4
    'epochs': 2,
    'folds': 2,
    'lr': 1e-3,
    'weight_decay': 1e-2,
    'pos_weight': 20,
    'train_dir': r'C:\Users\maila\Desktop\Physionet-Data\train',
    'output_dir': r'C:\Users\maila\Desktop\Physionet-Data\Pre_Processed_train',
    'csv_path': r'C:\Users\maila\Desktop\Physionet-Data\train.csv'
}

np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

os.makedirs(CONFIG['output_dir'], exist_ok=True)

In [2]:
def load_and_resize_image(image_path):
    """Load image and resize to target dimension."""
    image = cv2.imread(image_path)
    if image is None:
        return None
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (CONFIG['image_size'][0], CONFIG['image_size'][1]), interpolation=cv2.INTER_LINEAR)
    return image[:CONFIG['crop_size'][1], :CONFIG['crop_size'][0]]

def get_series_signals(csv_path):
    """Combine 12 leads into 4 series according to challenge standards."""
    try:
        df = pd.read_csv(csv_path)
        # Handle rhythm lead fallback
        if 'II-rhythm' not in df.columns:
            df['II-rhythm'] = df['II']
        
        # Ensure II is NaN where I is NaN (if applicable)
        if 'I' in df.columns:
            df.loc[df['I'].isna(), 'II'] = np.nan
        df.fillna(0, inplace=True)
        
        series0 = (df['I'] + df['aVR'] + df['V1'] + df['V4']).values
        series1 = (df['II'] + df['aVL'] + df['V2'] + df['V5']).values
        series2 = (df['III'] + df['aVF'] + df['V3'] + df['V6']).values
        series3 = df['II-rhythm'].values
        
        return np.stack([series0, series1, series2, series3])
    except Exception as e:
        print(f"Error processing CSV {csv_path}: {e}")
        return None

def signal_to_mask(series_signals, shape):
    """Convert signal values to a pixel-level mask."""
    H, W = shape
    mask = np.zeros((4, H, W), dtype=np.float32)
    
    for i in range(4):
        signal = series_signals[i]
        # Map time to x, signal to y
        for t, val in enumerate(signal):
            if t >= W - CONFIG['t0']: break
            x = CONFIG['t0'] + t
            y = int(CONFIG['zero_mv'][i] - val * CONFIG['mv_to_pixel'])
            if 0 <= y < H:
                mask[i, y, x] = 1.0
    return mask


In [3]:
def preprocess_dataset():
    """Preprocess all patients and save processed images and masks."""
    patient_ids = [d for d in os.listdir(CONFIG['train_dir']) if os.path.isdir(os.path.join(CONFIG['train_dir'], d))]
    print(f"Found {len(patient_ids)} patients. Starting preprocessing...")
    
    for pid in tqdm(patient_ids):
        pid_dir = os.path.join(CONFIG['train_dir'], pid)
        out_pid_dir = os.path.join(CONFIG['output_dir'], pid)
        os.makedirs(out_pid_dir, exist_ok=True)
        
        # Load image
        img_files = [f for f in os.listdir(pid_dir) if f.endswith('.png')]
        if not img_files: continue
        image = load_and_resize_image(os.path.join(pid_dir, img_files[0]))
        
        # Load signal
        csv_files = [f for f in os.listdir(pid_dir) if f.endswith('.csv')]
        if not csv_files: continue
        signals = get_series_signals(os.path.join(pid_dir, csv_files[0]))
        
        if image is not None and signals is not None:
            mask = signal_to_mask(signals, image.shape[:2])
            
            # Save processed data
            cv2.imwrite(os.path.join(out_pid_dir, 'image.png'), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
            np.savez_compressed(os.path.join(out_pid_dir, 'mask.npz'), mask=mask)
            # Copy original CSV for reference
            pd.read_csv(os.path.join(pid_dir, csv_files[0])).to_csv(os.path.join(out_pid_dir, 'original.csv'), index=False)

# Run preprocessing
# preprocess_dataset() # Uncomment to run full preprocessing


In [4]:
class ECGSeriesDataset(Dataset):
    """Dataset that crops 4 strips for the series model."""
    def __init__(self, patient_ids, base_dir, transform=None, is_train=True):
        self.patient_ids = patient_ids
        self.base_dir = base_dir
        self.transform = transform
        self.is_train = is_train
        self.window_size = CONFIG['window_size']
        self.zero_mv = CONFIG['zero_mv']

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        pid_dir = os.path.join(self.base_dir, pid)
        
        image = cv2.imread(os.path.join(pid_dir, 'image.png'))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = np.load(os.path.join(pid_dir, 'mask.npz'))['mask']
        
        if self.transform:
            # Albumentations expects (H, W, C)
            mask_t = mask.transpose(1, 2, 0)
            augmented = self.transform(image=image, mask=mask_t)
            image = augmented['image']
            mask = augmented['mask'].transpose(2, 0, 1)

        images, masks = [], []
        H, W, _ = image.shape
        for i, zmv in enumerate(self.zero_mv):
            h0, h1 = int(zmv) - self.window_size, int(zmv) + self.window_size
            src_h0, src_h1 = max(0, h0), min(H, h1)
            dst_h0, dst_h1 = src_h0 - h0, src_h0 - h0 + (src_h1 - src_h0)
            
            strip_img = np.zeros((self.window_size*2, W, 3), dtype=np.uint8)
            strip_img[dst_h0:dst_h1] = image[src_h0:src_h1]
            images.append(strip_img)
            
            strip_mask = np.zeros((self.window_size*2, W), dtype=np.float32)
            strip_mask[dst_h0:dst_h1] = mask[i][src_h0:src_h1]
            masks.append(strip_mask)
            
        images = np.stack(images).transpose(0, 3, 1, 2) # (4, 3, H, W)
        masks = np.expand_dims(np.stack(masks), 1) # (4, 1, H, W)
        
        return {
            'image': torch.from_numpy(images).float() / 255.0,
            'pixel': torch.from_numpy(masks).float(),
            'id': pid
        }

def get_transforms(is_train=True):
    if is_train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            ToTensorV2()
        ])
    return A.Compose([ToTensorV2()])


In [5]:
class CrossSeriesFusion(nn.Module):
    """Fuses features across different ECG strips."""
    def __init__(self, channels, num_series=4):
        super().__init__()
        self.num_series = num_series
        self.mix = nn.Sequential(
            nn.Conv2d(channels * num_series, channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x, batch_size):
        # x: (B*4, C, H, W)
        C, H, W = x.shape[1:]
        x_series = x.view(batch_size, self.num_series, C, H, W)
        concat = x_series.transpose(1, 2).reshape(batch_size, C * self.num_series, H, W)
        mixed = self.mix(concat).unsqueeze(1).expand(-1, self.num_series, -1, -1, -1)
        fused = x_series + mixed
        return fused.reshape(batch_size * self.num_series, C, H, W)

class ECGDigitizer(nn.Module):
    def __init__(self, encoder='resnet34', weights='imagenet'):
        super().__init__()
        model = smp.Unet(encoder_name=encoder, encoder_weights=weights, in_channels=3, classes=1)
        self.encoder = model.encoder
        self.decoder = model.decoder
        self.fusion = CrossSeriesFusion(512) # Based on resnet34 bottleneck
        self.head = nn.Conv2d(16, 1, kernel_size=1)
        self.dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, batch):
        img = batch['image'].to(CONFIG['device'])
        batch_size, num_series, C, H, W = img.shape
        x = img.view(batch_size * num_series, C, H, W)
        
        features = self.encoder(x)
        features = list(features)
        
        # Fuse high-level features
        features[-1] = self.fusion(features[-1], batch_size)
        
        # Pass features as a list/tuple, not unpacked, to match decoder's expected signature
        decoder_out = self.decoder(features)
        pixel_logits = self.head(decoder_out).view(batch_size, num_series, 1, H, W)
        
        output = {'logits': pixel_logits}
        if 'pixel' in batch:
            target = batch['pixel'].to(CONFIG['device'])
            bce = F.binary_cross_entropy_with_logits(
                pixel_logits, target, pos_weight=torch.tensor([CONFIG['pos_weight']]).to(CONFIG['device'])
            )
            dice = self.dice_loss(pixel_logits, target)
            output['loss'] = bce + dice
            
        return output

In [6]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc='Training', leave=False)
    optimizer.zero_grad()
    
    for i, batch in enumerate(pbar):
        try:
            with autocast(device_type=CONFIG['device'], enabled=(CONFIG['device']=='cuda')):
                out = model(batch)
                loss = out['loss'] / CONFIG['accumulation_steps']
            
            if CONFIG['device'] == 'cuda':
                scaler.scale(loss).backward()
            else:
                loss.backward()
            
            if (i + 1) % CONFIG['accumulation_steps'] == 0:
                if CONFIG['device'] == 'cuda':
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
            
            total_loss += loss.item() * CONFIG['accumulation_steps']
            pbar.set_postfix({'loss': f"{loss.item() * CONFIG['accumulation_steps']:.4f}"})
            
            # Immediate memory cleanup
            del out, loss
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print("| WARNING: CUDA Out of Memory during batch. Clearing cache...")
                torch.cuda.empty_cache()
                gc.collect()
                continue
            else:
                raise e
        
    return total_loss / len(loader)

def validate(model, loader):
    model.eval()
    total_loss = 0
    pbar = tqdm(loader, desc='Validating', leave=False)
    with torch.no_grad():
        for batch in pbar:
            try:
                out = model(batch)
                total_loss += out['loss'].item()
                del out
            except RuntimeError as e:
                if "out of memory" in str(e):
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise e
    return total_loss / len(loader)

def calculate_metrics(pred, truth):
    """Calculate SNR, RMSE, MAE, MSE."""
    mse = np.mean((pred - truth)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred - truth))
    signal_power = np.mean(truth**2)
    snr = 10 * np.log10(signal_power / (mse + 1e-7))
    return {'SNR': snr, 'RMSE': rmse, 'MAE': mae, 'MSE': mse}

def mask_to_signal(pixel_prob):
    """Convert pixel map back to series signals."""
    # pixel_prob: (4, 1, H, W) -> (4, H, W)
    pixel_prob = pixel_prob.squeeze(1)
    H, W = pixel_prob.shape[1:]
    signals = []
    for i in range(4):
        p = pixel_prob[i]
        # Find strongest pixel in each column
        amax = p.argmax(0) # Index of max probability
        # Convert pixel index back to mV
        s = (CONFIG['zero_mv'][i] - amax) / CONFIG['mv_to_pixel']
        # Crop to signal region
        s_cropped = s[CONFIG['t0']:CONFIG['t1']]
        signals.append(s_cropped)
    return np.stack(signals)

In [7]:
# Main Execution
patient_ids = [pid for pid in os.listdir(CONFIG['output_dir']) if os.path.isdir(os.path.join(CONFIG['output_dir'], pid))]
print(f"Total patients found for training: {len(patient_ids)}")

train_pids, test_pids = train_test_split(patient_ids, test_size=0.3, random_state=42)

kf = KFold(n_splits=CONFIG['folds'], shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(train_pids)):
    print(f"\n{'='*20} Fold {fold} {'='*20}")
    
    # Reset device at start of fold
    CONFIG['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    torch.cuda.empty_cache()
    gc.collect()
    
    fold_train_pids = [train_pids[i] for i in train_idx]
    fold_val_pids = [train_pids[i] for i in val_idx]
    
    train_ds = ECGSeriesDataset(fold_train_pids, CONFIG['output_dir'], is_train=True)
    val_ds = ECGSeriesDataset(fold_val_pids, CONFIG['output_dir'], is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)
    
    model = ECGDigitizer().to(CONFIG['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'] * len(train_loader))
    scaler = torch.amp.GradScaler(enabled=(CONFIG['device'] == 'cuda'))
    
    try:
        epoch_pbar = tqdm(range(CONFIG['epochs']), desc=f'Fold {fold} Epochs')
        for epoch in epoch_pbar:
            train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
            val_loss = validate(model, val_loader)
            print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
            
            torch.cuda.empty_cache()
            gc.collect()
            
    except RuntimeError as e:
        if "out of memory" in str(e):
            print("\n" + "!"*60)
            print("CRITICAL: GPU OUT OF MEMORY even at batch size 1.")
            print("FALLING BACK TO CPU. Training will be VERY SLOW.")
            print("!"*60 + "\n")
            
            # Fallback to CPU
            CONFIG['device'] = 'cpu'
            model = model.to('cpu')
            torch.cuda.empty_cache()
            gc.collect()
            
            # Restart loops on CPU (simplified)
            for epoch in range(CONFIG['epochs']):
                train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
                val_loss = validate(model, val_loader)
                print(f"Epoch {epoch} (CPU): Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
        else:
            raise e
    
    # Save fold model
    torch.save(model.state_dict(), f'model_fold{fold}.pth')
    
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

Total patients found for training: 374

==================== Fold 0 ====================


Fold 0 Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/130 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Final Evaluation on Test Set
model.load_state_dict(torch.load('model_fold0.pth')) # Load first fold for demo
test_ds = ECGSeriesDataset(test_pids, CONFIG['output_dir'], is_train=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

results = []
metrics_list = []

model.eval()
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= 5: break # Evaluate first 5
        out = model(batch)
        pred_mask = torch.sigmoid(out['logits']).cpu().numpy()[0]
        pred_signal = mask_to_signal(pred_mask)
        
        # Load truth
        pid = batch['id'][0]
        truth_csv = os.path.join(CONFIG['train_dir'], pid, f"{pid}.csv")
        truth_signal = get_series_signals(truth_csv)
        # Truth might be longer, resample or crop truth to 5000 samples
        truth_signal = truth_signal[:, :5000]
        
        m = calculate_metrics(pred_signal, truth_signal)
        metrics_list.append(m)
        results.append((truth_signal, pred_signal, pid))

# Print Average Metrics
avg_metrics = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0].keys()}
print("\nTest Set Metrics (Average of sample):")
for k, v in avg_metrics.items():
    print(f"{k}: {v:.4f}")

# Plot one sample
truth, pred, pid = results[0]
plt.figure(figsize=(15, 10))
for i in range(4):
    plt.subplot(4, 1, i+1)
    plt.plot(truth[i], label='Ground Truth', alpha=0.7)
    plt.plot(pred[i], label='Predicted', alpha=0.7, linestyle='--')
    plt.title(f"Series {i} - Patient {pid}")
    plt.legend()
plt.tight_layout()
plt.show()
